# PatchCore Training (Memory Bank Construction)

Builds PatchCore memory banks for MVTec AD anomaly detection.

**Key Concept:** PatchCore (Roth et al., CVPR 2022) achieves state-of-the-art anomaly detection
using features from a **pretrained ImageNet model** — no gradient-based training required.

**How it works:**
1. Extract patch-level features from a pretrained ResNet backbone
2. Build a "memory bank" of normal patch features (from training data)
3. At test time, find nearest neighbor distance for each patch
4. High distance = anomaly

**Advantages over autoencoder-based methods:**
- No gradient training required (only feature extraction)
- Uses rich ImageNet features instead of learning from scratch
- Typically achieves higher AUC than CAE/VAE/DAE

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import sys
sys.path.insert(0, 'F:/Thesis')

import torch
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import numpy as np
import pandas as pd
import time
from sklearn.metrics import roc_auc_score, roc_curve

from src.config import DEVICE, MODELS_DIR, FIGURES_DIR, OUTPUTS_DIR, MVTEC_CATEGORIES, ensure_dirs
from src.data import create_mvtec_dataloaders
from src.data.transforms import denormalize
from src.models.patchcore import create_patchcore

ensure_dirs()
print(f"Device: {DEVICE}")

## Configuration

In [ ]:
CONFIG = {
    'batch_size': 8,
    'backbone': 'resnet18',
    'k': 3,
    'subsample_ratio': 0.1,
}

CATEGORIES_TO_TRAIN = ['bottle']
# CATEGORIES_TO_TRAIN = MVTEC_CATEGORIES  # Uncomment to train all

print(f"Categories to train: {CATEGORIES_TO_TRAIN}")
print(f"Backbone: {CONFIG['backbone']}")
print(f"k-NN: {CONFIG['k']}")
print(f"Subsample ratio: {CONFIG['subsample_ratio']}")

## Model Overview

In [ ]:
model_preview = create_patchcore(
    backbone=CONFIG['backbone'],
    k=CONFIG['k'],
    subsample_ratio=CONFIG['subsample_ratio']
)

total_params = sum(p.numel() for p in model_preview.parameters())
trainable_params = sum(p.numel() for p in model_preview.parameters() if p.requires_grad)

print(f"PatchCore Model:")
print(f"  Backbone: {CONFIG['backbone']}")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,} (all frozen!)")
print(f"  Feature extraction layers: layer2, layer3")
del model_preview

## Visualization Helpers

In [ ]:
def save_and_show(fig, path):
    """Save figure to disk and display it."""
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    print(f"  Saved: {path}")


def plot_anomaly_maps(model, test_loader, category, device, save_path, n_samples=6):
    """Plot original, anomaly map, and heatmap overlay for test samples."""
    model.eval()
    # Gather some normal and some anomalous samples
    all_imgs, all_masks, all_labels = [], [], []
    for img, mask, label in test_loader:
        all_imgs.append(img)
        all_masks.append(mask)
        all_labels.append(label)
    all_imgs = torch.cat(all_imgs, dim=0)
    all_masks = torch.cat(all_masks, dim=0)
    all_labels = torch.cat(all_labels, dim=0)

    # Select a mix of normal and anomalous
    normal_idx = (all_labels == 0).nonzero(as_tuple=True)[0]
    anomaly_idx = (all_labels == 1).nonzero(as_tuple=True)[0]
    n_norm = min(n_samples // 2, len(normal_idx))
    n_anom = min(n_samples - n_norm, len(anomaly_idx))
    selected = torch.cat([normal_idx[:n_norm], anomaly_idx[:n_anom]])
    n = len(selected)
    if n == 0:
        print("  No samples available for visualization.")
        return

    imgs = all_imgs[selected].to(device)
    lbls = all_labels[selected]

    with torch.no_grad():
        anomaly_maps = model.get_anomaly_map(imgs)
        scores = model.get_anomaly_score(imgs)

    fig, axes = plt.subplots(n, 3, figsize=(12, 4 * n))
    if n == 1:
        axes = axes[np.newaxis, :]

    for i in range(n):
        orig = denormalize(imgs[i].cpu()).permute(1, 2, 0).numpy().clip(0, 1)
        amap = anomaly_maps[i, 0].cpu().numpy()
        amap_norm = (amap - amap.min()) / (amap.max() - amap.min() + 1e-8)
        lbl = 'Anomaly' if lbls[i].item() == 1 else 'Normal'
        sc = scores[i].item()

        axes[i, 0].imshow(orig)
        axes[i, 0].set_title(f'Original [{lbl}]', fontsize=10)
        axes[i, 0].axis('off')

        axes[i, 1].imshow(amap_norm, cmap='hot')
        axes[i, 1].set_title(f'Anomaly Map (score={sc:.4f})', fontsize=10)
        axes[i, 1].axis('off')

        axes[i, 2].imshow(orig)
        axes[i, 2].imshow(amap_norm, cmap='jet', alpha=0.5)
        axes[i, 2].set_title('Heatmap Overlay', fontsize=10)
        axes[i, 2].axis('off')

    fig.suptitle(f'PatchCore Anomaly Maps — {category}', fontsize=16, fontweight='bold')
    plt.tight_layout()
    save_and_show(fig, save_path)


def plot_score_distribution(all_scores, all_labels, category, save_path):
    """Plot anomaly score distribution for normal vs anomalous."""
    scores_arr = np.array(all_scores)
    labels_arr = np.array(all_labels)
    normal_scores = scores_arr[labels_arr == 0]
    anomaly_scores = scores_arr[labels_arr == 1]

    fig, ax = plt.subplots(figsize=(8, 5))
    if len(normal_scores) > 0:
        ax.hist(normal_scores, bins=30, alpha=0.6, label=f'Normal (n={len(normal_scores)})', color='green')
    if len(anomaly_scores) > 0:
        ax.hist(anomaly_scores, bins=30, alpha=0.6, label=f'Anomaly (n={len(anomaly_scores)})', color='red')
    ax.set_xlabel('Anomaly Score')
    ax.set_ylabel('Count')
    ax.set_title(f'PatchCore Score Distribution — {category}', fontsize=14, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    save_and_show(fig, save_path)


def plot_roc_curve(all_scores, all_labels, auc_val, category, save_path):
    """Plot ROC curve."""
    fpr, tpr, _ = roc_curve(all_labels, all_scores)
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot(fpr, tpr, 'b-', linewidth=2, label=f'PatchCore (AUC = {auc_val:.4f})')
    ax.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5, label='Random')
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(f'ROC Curve — {category}', fontsize=14, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1])
    save_and_show(fig, save_path)


def plot_memory_bank_stats(model, category, save_path):
    """Visualize memory bank statistics (feature distribution)."""
    if model.memory_bank is None:
        print("  No memory bank to visualize.")
        return

    mb = model.memory_bank.numpy()
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # Feature magnitude distribution
    norms = np.linalg.norm(mb, axis=1)
    axes[0].hist(norms, bins=50, color='steelblue', alpha=0.7)
    axes[0].set_xlabel('L2 Norm')
    axes[0].set_ylabel('Count')
    axes[0].set_title('Feature Magnitude Distribution')
    axes[0].grid(True, alpha=0.3)

    # Feature variance per dimension (first 100 dims)
    var_per_dim = np.var(mb, axis=0)
    axes[1].bar(range(min(100, len(var_per_dim))), var_per_dim[:100], color='coral', alpha=0.7)
    axes[1].set_xlabel('Feature Dimension')
    axes[1].set_ylabel('Variance')
    axes[1].set_title('Variance per Dimension (first 100)')
    axes[1].grid(True, alpha=0.3)

    # Summary stats
    info_text = (
        f"Memory Bank Stats\n"
        f"{'—' * 25}\n"
        f"Patches: {mb.shape[0]:,}\n"
        f"Feature dim: {mb.shape[1]}\n"
        f"Mean norm: {norms.mean():.2f}\n"
        f"Std norm: {norms.std():.2f}\n"
        f"Min norm: {norms.min():.2f}\n"
        f"Max norm: {norms.max():.2f}"
    )
    axes[2].text(0.1, 0.5, info_text, transform=axes[2].transAxes,
                 fontsize=12, verticalalignment='center', fontfamily='monospace',
                 bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
    axes[2].set_title('Summary')
    axes[2].axis('off')

    fig.suptitle(f'PatchCore Memory Bank — {category}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    save_and_show(fig, save_path)

## Memory Bank Construction Function

In [ ]:
def build_patchcore_category(category):
    print(f"\n{'='*60}")
    print(f"Building PatchCore Memory Bank: {category.upper()}")
    print(f"{'='*60}")

    # 1. Load Data
    try:
        train_loader, test_loader = create_mvtec_dataloaders(
            category, batch_size=CONFIG['batch_size'], return_mask=True
        )
    except Exception as e:
        print(f"Skipping {category}: {e}")
        return None

    print(f"  Train batches: {len(train_loader)}")
    print(f"  Test batches:  {len(test_loader)}")

    # 2. Create Model
    model = create_patchcore(
        backbone=CONFIG['backbone'],
        k=CONFIG['k'],
        subsample_ratio=CONFIG['subsample_ratio']
    )

    # 3. Build Memory Bank
    start_time = time.time()
    model.fit(train_loader, device=DEVICE)
    fit_time = time.time() - start_time
    print(f"  Memory bank built in {fit_time:.2f}s")

    # 4. Evaluate
    model.eval()
    all_scores, all_labels = [], []

    print("  Evaluating on test set...")
    with torch.no_grad():
        for img, mask, label in test_loader:
            img = img.to(DEVICE)
            scores = model.get_anomaly_score(img)
            all_scores.extend(scores.cpu().numpy())
            all_labels.extend(label.numpy())

    try:
        auc = roc_auc_score(all_labels, all_scores)
        print(f"  {category.upper()} ROC-AUC: {auc:.4f}")
    except:
        auc = 0.0
        print(f"  Could not compute AUC")

    # 5. Save memory bank
    save_path = MODELS_DIR / f'patchcore_{category}_memory.pth'
    model.save_memory_bank(str(save_path))
    print(f"  Memory bank saved: {save_path}")

    # 6. Generate all visualizations
    print("  Generating visualizations...")

    # 6a. Memory bank statistics
    plot_memory_bank_stats(model, category,
                           FIGURES_DIR / f'patchcore_{category}_memory_stats.png')

    # 6b. Anomaly maps (mix of normal + anomalous samples)
    plot_anomaly_maps(model, test_loader, category, DEVICE,
                      FIGURES_DIR / f'patchcore_{category}_anomaly_maps.png')

    # 6c. Score distribution
    plot_score_distribution(all_scores, all_labels, category,
                            FIGURES_DIR / f'patchcore_{category}_scores.png')

    # 6d. ROC curve
    if auc > 0:
        plot_roc_curve(all_scores, all_labels, auc, category,
                       FIGURES_DIR / f'patchcore_{category}_roc.png')

    return {
        'category': category,
        'auc': auc,
        'fit_time_s': round(fit_time, 2),
        'memory_bank_size': model.memory_bank.shape[0] if model.memory_bank is not None else 0,
        'feature_dim': model.memory_bank.shape[1] if model.memory_bank is not None else 0,
    }

## Run Memory Bank Construction

In [ ]:
results = []

for category in CATEGORIES_TO_TRAIN:
    result = build_patchcore_category(category)
    if result is not None:
        results.append(result)

print(f"\n{'='*60}")
print("MEMORY BANK CONSTRUCTION COMPLETE")
print(f"{'='*60}")

## Results Summary

In [ ]:
if results:
    df = pd.DataFrame(results)
    df = df.sort_values('auc', ascending=False)
    print("\nPatchCore Results:")
    print(df.to_string(index=False))
    print(f"\nMean AUC: {df['auc'].mean():.4f}")

    # Save results table as CSV
    csv_path = OUTPUTS_DIR / 'patchcore_results.csv'
    df.to_csv(csv_path, index=False)
    print(f"Results saved: {csv_path}")

    # Bar chart of AUC per category
    if len(df) > 1:
        fig, ax = plt.subplots(figsize=(10, 5))
        colors = ['green' if a >= 0.7 else 'orange' if a >= 0.5 else 'red' for a in df['auc']]
        ax.barh(df['category'], df['auc'], color=colors)
        ax.set_xlabel('ROC-AUC')
        ax.set_title('PatchCore Performance by Category', fontsize=14, fontweight='bold')
        ax.set_xlim([0, 1])
        ax.axvline(x=0.5, color='gray', linestyle='--', alpha=0.5, label='Random')
        for i, v in enumerate(df['auc']):
            ax.text(v + 0.01, i, f'{v:.3f}', va='center', fontsize=9)
        ax.legend()
        ax.grid(True, alpha=0.3, axis='x')
        save_and_show(fig, FIGURES_DIR / 'patchcore_auc_summary.png')
else:
    print("No results to display.")